In [2]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3-client/"
os.environ["PYSPARK_PYTHON"] = "/opt/sdp/mlpy3811v23/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/opt/sdp/mlpy3811v23/bin/python"
os.environ["LD_LIBRARY_PATH"] = "/opt/python/virtualenv/jupyter/lib"
sys.path.insert(0, "/usr/sdp/current/spark3-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3-client/python/lib/py4j-0.10.9.3-src.zip")


import yaml
from functools import reduce
from itertools import chain
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from dateutil.relativedelta import relativedelta
import datetime
from collections import defaultdict
import pyspark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import MinMaxScaler

from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import MapType, StringType, IntegerType, DoubleType, ByteType
from pyspark.sql.functions import from_json
from pyspark.sql import SparkSession
from pyspark import SparkConf

from dataclasses import dataclass
from IPython.display import display, clear_output
from typing import List, Union, Callable
import subprocess
import time

from pathlib import Path
from git import Repo


session = (
    SparkSession.builder
        .appName("sequence_data_collection_example")
        .master("yarn")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.executor.instances", "10")
        .config("spark.driver.memory", "10g")
        .config("spark.executor.memory",  "20g")
        .config("spark.executor.cores", "4")
        .config("spark.driver.maxResultSize", "10g")
        .config("spark.hadoop.hive.exec.dynamic.partition", "true")
        .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
        .config("spark.shuffle.service.enabled", "true")
        .config("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.sql.broadcastTimeout", "300")
        .config("spark.port.maxRetries", "150")
        .config("spark.shuffle.memoryFraction", "0.5")
        .config("spark.sql.legacy.timeParserPolicy","LEGACY")
        .config("spark.kryoserializer.buffer.max", "1536")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
        .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
        .config("spark.hadoop.parquet.block.size", "134217728")
        .config("spark.sql.autoBroadcastJoinThreshold", "-1")
        .config("spark.scheduler.allocation.file", "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/utils/fairScheduleConfig.xml")
        .config("spark.scheduler.mode", "FAIR")
        .enableHiveSupport()
)

spark = session.getOrCreate()  
spark.sparkContext.setLogLevel("ERROR")

In [3]:
sys.path.insert(0, "../..")

from avatar.preprocessing.spark.pipeline import EventSequencePreprocessor

In [4]:
# Configuration
txn_path = "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/digital_traces/txn"
evk_union_vnv_path = "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/digital_traces/evk_union_vnv_txt"
bucket_interval = (1, 10)
txn_columns = {
    "categorical_columns": ["evt_attr_9"],  # MCC
    "numeric_columns": ["evt_attr_15"],  # Price
}
evk_vnv_columns = {
    "categorical_columns": ["channel_group", "sale_product_class_id"],
    "numeric_columns": [],
}

# Чтение транзакций
txn_df = (
    spark.read.parquet(
        txn_path
    )
    .select("epk_id", "evt_dttm", "evt_attr_9", "evt_attr_15", "bucket_num")
    .where(F.col("bucket_num").between(*bucket_interval))
    .withColumn("event_ids", F.lit(0))  # Добавляем идентификатор события
)
txn_df.show(2)


# Чтение витрины коммункаций, новых выдач 
evk_union_vnv_df = (
    spark.read.parquet(
        evk_union_vnv_path
    )
    .select("epk_id", "evt_dttm", "channel_group", "sale_product_class_id", "bucket_num")
    .where(F.col("bucket_num").between(*bucket_interval))
    .withColumn("event_ids", F.lit(1))  # Добавляем идентификатор события
)
evk_union_vnv_df.show(2)

# Объединенеие источников в один ДатаФрейм
unioned_df = txn_df.unionByName(
    evk_union_vnv_df,
    allowMissingColumns=True,
)

# Промежуточное подсохранение
unioned_df.write.parquet(
    "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/next_event_prediction/temp_unioned_df"
)

unioned_df = spark.read.parquet(
    "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/next_event_prediction/temp_unioned_df"
)
unioned_df.show(2)

+-------------------+-------------------+----------+-----------+----------+---------+
|             epk_id|           evt_dttm|evt_attr_9|evt_attr_15|bucket_num|event_ids|
+-------------------+-------------------+----------+-----------+----------+---------+
|1126087768713110519|2024-02-01 22:17:55|      9999|          5|         8|        0|
|1833357510141884868|2024-02-03 16:02:06|      5499|         44|         8|        0|
+-------------------+-------------------+----------+-----------+----------+---------+
only showing top 2 rows



+-------------------+----------+-------------+---------------------+----------+---------+
|             epk_id|  evt_dttm|channel_group|sale_product_class_id|bucket_num|event_ids|
+-------------------+----------+-------------+---------------------+----------+---------+
|1130200773189156699|2024-12-01|            3|                    1|         3|        1|
|1129398871580708668|2024-12-21|            2|                   20|         3|        1|
+-------------------+----------+-------------+---------------------+----------+---------+
only showing top 2 rows



+-------------------+-------------------+----------+-----------+----------+---------+-------------+---------------------+
|             epk_id|           evt_dttm|evt_attr_9|evt_attr_15|bucket_num|event_ids|channel_group|sale_product_class_id|
+-------------------+-------------------+----------+-----------+----------+---------+-------------+---------------------+
|1512065617662442640|2024-12-16 15:09:28|      4829|        211|         1|        0|         null|                 null|
|1845558064480567884|2024-12-09 11:10:13|         0|       1000|         1|        0|         null|                 null|
+-------------------+-------------------+----------+-----------+----------+---------+-------------+---------------------+
only showing top 2 rows



In [5]:
# Инициализация и применения препроцессора к данным

preprocessor = EventSequencePreprocessor(
    categorical_columns=txn_columns["categorical_columns"] + evk_vnv_columns["categorical_columns"],
    numeric_columns=txn_columns["numeric_columns"] + evk_vnv_columns["numeric_columns"],
    event_time_column="evt_dttm",
    id_column="epk_id",
    event_type_ids_column="event_ids",
    time_unit="days",
)

processed_df = preprocessor.fit_transform(df=unioned_df)
processed_df.write.parquet(
    "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/next_event_prediction/processed_sequences"
)

In [8]:
!hdfs dfs -rm -r -skipTrash hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/next_event_prediction/temp_unioned_df

Deleted hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/next_event_prediction/temp_unioned_df


In [9]:
# Информация о каждом атрибуте для обучения

preprocessor.columns_meta

{'evt_attr_9': {'type': 'categorical', 'n_classes': 524},
 'channel_group': {'type': 'categorical', 'n_classes': 10},
 'sale_product_class_id': {'type': 'categorical', 'n_classes': 39},
 'evt_attr_15': {'type': 'numeric', 'n_classes': 1}}

In [12]:
import torch


k = 2
targets = torch.tensor([1, 2, 3, 5, 6]).long()

random_tensor = torch.randint(1, k + 1, size=targets.size())
incorrect_dropped_token_indices = (random_tensor == k).int()
correct_dropped_token_indices = (random_tensor == k)

print(f"{incorrect_dropped_token_indices=}")
print(f"{correct_dropped_token_indices=}")

incorrect_masked_targets = targets.clone()
incorrect_masked_targets[incorrect_dropped_token_indices] = -100

correct_masked_targets = targets.clone()
correct_masked_targets[correct_dropped_token_indices] = -100

print(f"{incorrect_masked_targets=}") # if k == 2 only 0 and 1 indeces could be dropped
print(f"{correct_masked_targets=}")

incorrect_dropped_token_indices=tensor([1, 0, 0, 0, 1], dtype=torch.int32)
correct_dropped_token_indices=tensor([ True, False, False, False,  True])
incorrect_masked_targets=tensor([-100, -100,    3,    5,    6])
correct_masked_targets=tensor([-100,    2,    3,    5, -100])


In [ ]:
# Сохранение препроцессора

with open("artifacts/sequence_preprocessor.yaml", "w") as f:
    yaml.dump(preprocessor.dump(), f)

preprocessor.dump()